# DA6401 Report - Section 2.8 Error Analysis

Generate:
- Confusion matrix for the best model
- Creative failure visualization (misclassification gallery)
- Top confusion-pair bar chart


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import wandb
from sklearn.metrics import confusion_matrix

import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from ann.neural_network import NeuralNetwork
from utils.data_loader import load_dataset

print('Project root:', ROOT)


In [ ]:
RUN_EXPERIMENT = False  # Set True to run 2.8 and log a new W&B summary run.

PROJECT = 'da6401_assignment_1'
ENTITY = 'da25s007-'
OUT_DIR = ROOT / 'src' / 'tmp'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = ROOT / 'src' / 'best_model.npy'
BEST_CONFIG_PATH = ROOT / 'src' / 'best_config.json'


def load_model(model_path: Path):
    return np.load(model_path, allow_pickle=True).item()


def make_args_from_config(cfg: dict):
    return SimpleNamespace(
        dataset=cfg.get('dataset', 'mnist'),
        epochs=int(cfg.get('epochs', 1)),
        batch_size=int(cfg.get('batch_size', 64)),
        loss=cfg.get('loss', 'cross_entropy'),
        optimizer=cfg.get('optimizer', 'rmsprop'),
        learning_rate=float(cfg.get('learning_rate', 0.001)),
        weight_decay=float(cfg.get('weight_decay', 0.0)),
        num_layers=int(cfg.get('num_layers', 2)),
        hidden_size=cfg.get('hidden_size', [128, 128]),
        activation=cfg.get('activation', 'relu'),
        weight_init=cfg.get('weight_init', 'xavier'),
        seed=int(cfg.get('seed', 42)),
    )


def plot_confusion_matrix(cm: np.ndarray, out_path: Path):
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title('2.8: Confusion Matrix (Best MNIST model)')
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_xticks(np.arange(10))
    ax.set_yticks(np.arange(10))
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            v = int(cm[i, j])
            if v > 0:
                ax.text(j, i, str(v), ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches='tight')
    plt.close(fig)


def plot_failure_gallery(X_test, y_true, y_pred, top_pairs, out_path: Path, samples_per_pair: int = 4):
    rows = len(top_pairs)
    cols = samples_per_pair
    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.0 * rows))
    if rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, (t, p, count) in enumerate(top_pairs):
        idx = np.where((y_true == t) & (y_pred == p))[0][:samples_per_pair]
        for c in range(cols):
            ax = axes[r, c]
            ax.axis('off')
            if c < len(idx):
                ax.imshow(X_test[idx[c]].reshape(28, 28), cmap='gray')
                ax.set_title(f't={t}, p={p}', fontsize=8)
        axes[r, 0].set_ylabel(f'{t}->{p} ({count})', fontsize=8)

    fig.suptitle('2.8: Failure gallery (top confusion pairs)', fontsize=12)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches='tight')
    plt.close(fig)


def plot_top_confusions(top_pairs, out_path: Path):
    labels = [f'{t}->{p}' for (t, p, _) in top_pairs]
    counts = [int(c) for (_, _, c) in top_pairs]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(labels, counts)
    ax.set_title('2.8: Most frequent confusion pairs')
    ax.set_xlabel('True -> Pred')
    ax.set_ylabel('Count')
    ax.grid(True, axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches='tight')
    plt.close(fig)


if RUN_EXPERIMENT:
    cfg = json.loads(BEST_CONFIG_PATH.read_text())
    args = make_args_from_config(cfg)

    data = load_dataset(dataset=args.dataset, seed=args.seed)
    model = NeuralNetwork(args)
    weights = load_model(BEST_MODEL_PATH)
    model.set_weights(weights)

    test_metrics = model.evaluate(data['X_test'], data['y_test'], batch_size=args.batch_size)
    y_true = data['y_test']
    y_pred = np.argmax(test_metrics['logits'], axis=1)

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(10))

    cm_no_diag = cm.copy()
    np.fill_diagonal(cm_no_diag, 0)
    pair_idx = np.dstack(np.unravel_index(np.argsort(cm_no_diag.ravel())[::-1], cm_no_diag.shape))[0]
    top_pairs = []
    for i, j in pair_idx:
        c = int(cm_no_diag[i, j])
        if c <= 0:
            continue
        top_pairs.append((int(i), int(j), c))
        if len(top_pairs) == 6:
            break

    cm_path = OUT_DIR / 'report_2_8_confusion_matrix.png'
    gallery_path = OUT_DIR / 'report_2_8_failure_gallery.png'
    bar_path = OUT_DIR / 'report_2_8_top_confusions_bar.png'

    plot_confusion_matrix(cm, cm_path)
    plot_failure_gallery(data['X_test'], y_true, y_pred, top_pairs, gallery_path, samples_per_pair=4)
    plot_top_confusions(top_pairs, bar_path)

    run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name='report_2_8_summary',
        tags=['report', '2.8', 'error_analysis', 'summary'],
        config={'model_path': str(BEST_MODEL_PATH), 'config_path': str(BEST_CONFIG_PATH)},
    )

    conf_table = wandb.Table(columns=['true', 'pred', 'count'])
    for t, p, c in top_pairs:
        conf_table.add_data(t, p, c)

    run.log(
        {
            'analysis/confusion_matrix_plot': wandb.Image(str(cm_path)),
            'analysis/failure_gallery_plot': wandb.Image(str(gallery_path)),
            'analysis/top_confusions_bar_plot': wandb.Image(str(bar_path)),
            'analysis/top_confusions_table': conf_table,
        }
    )
    run.summary['test_accuracy'] = float(test_metrics['accuracy'])
    run.summary['test_f1'] = float(test_metrics['f1'])
    run.finish()

    payload = {
        'best_model': str(BEST_MODEL_PATH.relative_to(ROOT)),
        'best_config': str(BEST_CONFIG_PATH.relative_to(ROOT)),
        'test_accuracy': float(test_metrics['accuracy']),
        'top_confusion_pairs': [
            {'true': int(t), 'pred': int(p), 'count': int(c)} for (t, p, c) in top_pairs
        ],
        'artifacts': {
            'confusion_matrix_plot': str(cm_path.relative_to(ROOT)),
            'failure_gallery_plot': str(gallery_path.relative_to(ROOT)),
            'top_confusions_bar_plot': str(bar_path.relative_to(ROOT)),
        },
        'summary_run': 'report_2_8_summary',
    }

    out_json = OUT_DIR / 'report_2_8_error_analysis.json'
    out_json.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print('Wrote', out_json)
else:
    print('RUN_EXPERIMENT=False -> no W&B calls made.')


In [ ]:
summary_path = ROOT / 'src' / 'tmp' / 'report_2_8_error_analysis.json'
if summary_path.exists():
    data = json.loads(summary_path.read_text())
    print('Summary file:', summary_path)
    print('Best model:', data.get('best_model'))
    print('Test accuracy:', data.get('test_accuracy'))
    print('Top confusion pairs:')
    for row in data.get('top_confusion_pairs', []):
        print(' ', row)
else:
    print('No summary JSON yet. Set RUN_EXPERIMENT=True and run the previous cell.')
